In [1]:
import pandas as pd
import numpy as np

timestamps = pd.date_range(start="2025-03-01",periods=1000,freq="5T")
cpu_usage = np.random.uniform(low=20,high=80,size=1000)

data = pd.DataFrame({"timestamp":timestamps,"cpu_usage":cpu_usage})
data.to_csv("cpu_usage_data.csv",index=False)

<ipython-input-1-3e4201ab538b>:4: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  timestamps = pd.date_range(start="2025-03-01",periods=1000,freq="5T")


In [6]:
data = pd.read_csv("cpu_usage_data.csv")
data.head()

,timestamp,cpu_usage
0,2025-03-01 00:00:00,23.014591
1,2025-03-01 00:05:00,58.802717
2,2025-03-01 00:10:00,44.470909
3,2025-03-01 00:15:00,39.015316
4,2025-03-01 00:20:00,43.021348


In [9]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
data['cpu_usage_scaled'] = scaler.fit_transform(data['cpu_usage'].values.reshape(-1,1))
data.head()

,timestamp,cpu_usage,cpu_usage_scaled
0,2025-03-01 00:00:00,23.014591,0.049614
1,2025-03-01 00:05:00,58.802717,0.647330
2,2025-03-01 00:10:00,44.470909,0.407967
3,2025-03-01 00:15:00,39.015316,0.316850
4,2025-03-01 00:20:00,43.021348,0.383757


In [10]:
def create_sequences(data,seq_len):
  x,y=[],[]
  for i in range(len(data)-seq_len):
    x.append(data[i:i+seq_len])
    y.append(data[i+seq_len])
  return np.array(x),np.array(y)

seq_len=24

x,y=create_sequences(data['cpu_usage_scaled'],seq_len)


RNN MODEL

In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

model = Sequential()
model.add(SimpleRNN(50, input_shape=(seq_len, 1)))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [14]:
model.fit(x, y, epochs=20, batch_size=32)

Epoch 1/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.2400
Epoch 2/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0864
Epoch 3/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0877
Epoch 4/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0814
Epoch 5/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0819
Epoch 6/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0809
Epoch 7/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0790
Epoch 8/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0772
Epoch 9/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0787
Epoch 10/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0798
Epoch 11/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0843
Epoch 12/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0782
Epoch 13/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0811
Epoch 14/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0837
Epoch 15/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0760
Epoch 16/20
31/

In [15]:
model.save('cpu_predictor.h5')

In [17]:
!pip install deap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 3.5 MB/s eta 0:00:00


GA




In [24]:
import random
import numpy as np
from deap import base, creator, tools, algorithms

# Define fitness and individual
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))  # Minimize cost
creator.create("Individual", list, fitness=creator.FitnessMin)

# Initialize GA
toolbox = base.Toolbox()
toolbox.register("attr_int", random.randint, 1, 4)  # CPU cores (1 to 4)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_int, n=2)  # Individuals have 2 genes
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Define evaluation function
def evaluate(individual):
    total_cores = sum(individual)  # Sum of CPU cores
    predicted_usage = model.predict(np.array([x[-1]]))  # Use RNN prediction
    cost = total_cores * 10  # Cost per CPU core (e.g., $10 per core)
    if predicted_usage > 0.9:  # Penalize if usage exceeds 90%
        cost += 1000  # Large penalty
    return (cost,)

toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxTwoPoint)  # Use two-point crossover
toolbox.register("mutate", tools.mutUniformInt, low=1, up=4, indpb=0.2)
toolbox.register("select", tools.selTournament, tournsize=3)

# Run GA
population = toolbox.population(n=50)
algorithms.eaSimple(population, toolbox, cxpb=0.5, mutpb=0.2, ngen=10, verbose=True)

# Get the best solution
best_individual = tools.selBest(population, k=1)[0]
print("Optimal CPU cores:", best_individual)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step


/usr/local/lib/python3.11/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMin' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/usr/local/lib/python3.11/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━